In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Fixed blind nominal-latent synchronization — CPU only

Run all reuses the 18 saved recovered tensors from the fixed crop run
`inversion_crop_observation_20260918T171553283563Z`. No new generation, MP4,
VAE, Transformer, inversion or GPU work. Use a CPU runtime.

Search inputs are only each recovered tensor and the fixed public key/book.
All 14 nominal shifts (0–13) and both messages are scored using the SAME local
slices 1–31, two axes each. Raw amplitude times candidate state is averaged;
no direction normalization, oracle score, known start, fitted weight or threshold.
Keep all 28 scores and EPS ties, offset-maximized message margins and separate
offset peak gaps. Observer on/off remains auxiliary and never reranks candidates.
No exact RGB frame phase is estimated. OFF ranking is not positive detection.

All 18 blind JSON outputs persist BEFORE any attacker manifest or original
result is read for hash/provenance checks and truth join. Aligned starts report
nominal-shift accuracy. Nonaligned starts report floor/ceil and neighboring
scores separately, never select the more favorable map as success. Missing
and failed clips retain all 504 candidate slots. Crops from one video correlate.

Input is fixed under `MyDrive/Video-WM/InversionCropObservation/`.
New outputs, immutable source archive and launcher log are written under
`MyDrive/Video-WM/InversionBlindSync/inversion_blind_sync_<UTC>`.
No tuning, source overwrites, new model execution or scientific PASS.

Source SHA: 9669f794d7d474a7bc51893310fbd889cd2238ff.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util, json, os, signal, subprocess, sys
SOURCE_COMMIT = '9669f794d7d474a7bc51893310fbd889cd2238ff'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'inversion_blind_sync_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
if importlib.util.find_spec('numpy') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy'], check=True)


In [ ]:
INPUT = Path('/content/drive/MyDrive/Video-WM/InversionCropObservation/inversion_crop_observation_20260918T171553283563Z')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/InversionBlindSync') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.inversion_blind_sync_run', '--input', str(INPUT), '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({k:v for k,v in result['summary'].items() if k != 'per_clip'}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
